# 05 - Municipality harmonization

Checking the administrative crosswalk and the harmonization of municipalities on the canonical 2025 geography.

### Reproducibility

This notebook documents and verifies the territorial harmonization phase.

The transformations use the ISTAT data stored in `data/raw/` and are implemented in the scripts in the `scripts/` directory.

The notebook allows the normalization of ISTAT codes, administrative changes, and territorial comparability across different years to be checked.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

def project_root():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent]
    for c in candidates:
        if (c / "data" / "processed").exists() and (c / "metadata").exists():
            return c
    raise FileNotFoundError("Eseguire il notebook dalla root del repository o da notebooks/.")

ROOT = project_root()
ROOT

## Harmonization of municipalities 2019-2025
The canonical geography is ISTAT 2025. The crosswalk preserves mergers/incorporations and flags cases that cannot be reconstructed without primary fuzzy matching.

In [2]:
cw = pd.read_csv(ROOT / "metadata/municipality_crosswalk_2019_2025.csv", dtype=str, keep_default_na=False)
print(f"Relazioni crosswalk: {len(cw):,}")
print(f"Comuni correnti distinti: {cw['current_istat_code'].nunique():,}")
cw["aggregation_rule"].value_counts().to_frame("count")

Relazioni crosswalk: 7,955
Comuni correnti distinti: 7,896


,count
aggregation_rule,
sum,7953
not_reconstructible,2


In [3]:
nonrec = cw.loc[cw["aggregation_rule"] == "not_reconstructible"]
nonrec

,current_istat_code,current_name,predecessor_istat_code,predecessor_name,transformation_type,effective_date,source,aggregation_rule,notes
6590,081021,Trapani,081021,Trapani,partial_territorial_split_source,2021-02-20,https://www.istat.it/storage/codici-unita-ammi...,not_reconstructible,Parte del territorio di Trapani è stata scorpo...
6594,081025,Misiliscemi,081021,Trapani,partial_territorial_split_new_municipality,2021-02-20,https://www.istat.it/storage/codici-unita-ammi...,not_reconstructible,Comune istituito per scorporo di località da T...


In [4]:
ana = pd.read_csv(ROOT / "data/processed/analysis_municipality.csv", dtype=str, keep_default_na=False)
reggio = ana.loc[ana["province_istat_code"] == "080", "province"]
assert len(reggio) == 97
assert set(reggio) == {"Reggio di Calabria"}
print("OK: 97/97 righe della provincia 080 usano 'Reggio di Calabria'.")

OK: 97/97 righe della provincia 080 usano 'Reggio di Calabria'.
